Генерируем ссылки, по которым будем собирать адреса объявлений

In [42]:
base_link = "https://www.cian.ru/cat.php?deal_type=sale&engine_version=2&offer_type=flat&region=1&object_type%5B0%5D=1"

links = []

def generate_range(start, end, step):
    """Генерирует ссылки для диапазона цен"""
    curr = start
    while curr < end:
        #верхняя граница-1
        next_val = min(curr + step, end)
        max_p = next_val - 1
        
        url = f"{base_link}&minprice={curr}&maxprice={max_p}"
        links.append(url)
        
        curr = next_val
    print(len(links))
        
links.append('url')

#все до 6 млн берем вместе
generate_range(0, 6000000, 6000000)

#самый маленький шаг для минимума
generate_range(6_000_000, 25_000_000, 500_000)

#квартриры подороже - шаг 1 млн
generate_range(25_000_000, 50_000_000, 1_000_000)

#элита - шаг 5 млн
generate_range(50_000_000, 200_000_000, 5_000_000)

#элита покруче - шаг 50 млн
generate_range(200_000_000, 1_000_000_000, 50_000_000)


#дописываю остаток от 1 млрд
url = f"{base_link}&minprice=1000000000"
links.append(url)

#сохраняем полученные файлы
filename = "links.csv"
with open(filename, "w", encoding="utf-8") as f:
    for link in links:
        f.write(link + "\n")

print(f"Сгенерировано {len(links)-1} ссылок.")
print(f"Сохранено в файл: {filename}")

2
40
65
95
111
Сгенерировано 112 ссылок.
Сохранено в файл: links.csv


Тут определяем, сколько страниц для каждой ссылки нужно

In [43]:
import time
import pandas as pd
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import re
import math #для округлвверх

FILENAME = "links.csv"
OUTPUT_FILE = "checked_links.csv"

options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
results = []

def save_data(data_list, filename):
    if not data_list:
        return
    df = pd.DataFrame(data_list)
    header = not os.path.exists(filename)
    df.to_csv(filename, mode='a', index=False, header=header, encoding='utf-8-sig')
    print(f"Сохранено {len(data_list)} новых записей в {filename}")

try:
    df = pd.read_csv(FILENAME)
    links = df['url'].dropna().tolist()
    print(f"Всего ссылок в файле: {len(links)}")

    for i, url in enumerate(links):
        driver.get(url)
        time.sleep(1)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        header = soup.find('div', {'data-name': 'SummaryHeader'})
        if header:
            text = header.get_text()
            #регуляркой ищем "Найдено N объявлений"
            match = re.search(r'Найдено\s+([\d\s]+)', text)
            if match:
                clean_number = match.group(1).replace(' ', '').replace('\xa0', '')
        else:
            clean_number = 0
        results.append({
                'url': url,
                'total_ads': clean_number,
                'cnt_pages': math.ceil(int(clean_number)/28)
            })
        
        if (i+1) % 10 == 0:
            pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
            print(f'сохранено {i+1} ссылок.')

finally:
    driver.quit()
    #итог
    final_df = pd.DataFrame(results)
    final_df.to_csv(OUTPUT_FILE, index=False)
    print('все ссылки обработаны')

Всего ссылок в файле: 111
сохранено 10 ссылок.
сохранено 20 ссылок.
сохранено 30 ссылок.
сохранено 40 ссылок.
сохранено 50 ссылок.
сохранено 60 ссылок.
сохранено 70 ссылок.
сохранено 80 ссылок.
сохранено 90 ссылок.
сохранено 100 ссылок.
сохранено 110 ссылок.
все ссылки обработаны


### Собираем адреса объявлений

In [111]:
import time
import pandas as pd
import os
import winsound
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

#Настройки браузера

target_links = pd.read_csv("checked_links.csv")  # Файл с диапазонами и кол-вом страниц
FILENAME = "cian_final_data.csv" 
'''
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
'''
#настройки браузера
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--log-level=3")
options.page_load_strategy = 'eager'

#пытаемся обойти блокировку vpn (это не сработало)
#options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

prefs = {"profile.managed_default_content_settings.images": 2}
options.add_experimental_option("prefs", prefs)


driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

results = []

#-------------БЛОК ФУНКЦИЙ------------------------

#сохраняет данные и выводит об этом инфу
def save_data(data_list, filename):
    if not data_list:
        return
    df = pd.DataFrame(data_list)
    header = not os.path.exists(filename)
    df.to_csv(filename, mode='a', index=False, header=header, encoding='utf-8-sig')
    print(f"   -> Сохранено {len(data_list)} новых записей в {filename}")


#-------------ОСНОВНОЙ БЛОК------------------------
try:
    for i, row in target_links.iterrows():
        base_url = row['url']
        max_pages = int(row['cnt_pages']) 
        
        #скип пустых диапазонов (вдруг)
        if max_pages == 0:
            continue
            
        print(f"\n>>> Ссылка [{i+1}/{len(target_links)}] (страниц: {max_pages})")

        # цикл по страницам внутри ссылки
        for page_num in range(1, max_pages + 1):
            
            #ссылка для перехода
            if page_num == 1:
                current_url = base_url
            else:
                current_url = base_url + f"&p={page_num}"
            
            print(f"    страница {page_num} из {max_pages}")
            
            try: 
                #print(f"Вход: {current_url}")
                driver.get(current_url)
                
                #время на загрузку страницы, на всякий случай
                time.sleep(2)
            
                #постранично обрабатываем все доступные по запросу страницы (сверху фикс - были взяты все вторички Москвы)
                #print(f"\n--- Обработка страницы {page_num} ---")
                
                soup = BeautifulSoup(driver.page_source, 'html.parser')
                offers = soup.find_all('article')
        
                for offer in offers:
                    try:
                        #цена объекта
                        price_text = "0"
                        price_elem = offer.find('span', {'data-mark': 'MainPrice'})
                        if price_elem:
                            price_text = price_elem.get_text(strip=True)
                        else:
                            candidates = offer.find_all(string=lambda t: '₽' in t if t else False)
                            if candidates:
                                price_text = candidates[0]
        
                        clean_price = "".join([c for c in price_text if c.isdigit()])
                        
                        #ссылка на объявление
                        link_tag = offer.find('a', href=True)
                        link = link_tag['href'] if link_tag else "Нет ссылки"
        
                        #адрес
                        geo_labels = offer.find_all('a', {'data-name': 'GeoLabel'})
                        address = ", ".join([g.text.strip() for g in geo_labels])

                        
                        if not address:
                            title_div = offer.find('div', {'data-name': 'GeneralInfoSectionRowComponent'})
                            address = title_div.get_text(strip=True) if title_div else "Адрес не найден"

                        #аукцион
                        auction_tag = offer.find(string=re.compile(r"Аукцион"))
                        if auction_tag:
                            marker_text = True
                        else:
                            marker_text = False
                            
                        
                        results.append({
                            'price': clean_price,
                            'address': address,
                            'url': link,
                            'auction': marker_text
                        })
                    except Exception as e:
                        continue
                
                #----сохраняем инфу каждые 5 страниц---
                if page_num % 5 == 0:
                    save_data(results, FILENAME)
                    results = [] 
        
                
            except Exception as e:
                print(f"Ошибка при обработке страницы: {e}")
                time.sleep(5)
    time.sleep(1.5)

finally:
    if results:
        save_data(results, FILENAME)
    
    driver.quit()
    print("\nБраузер закрыт. Сбор завершен.")


>>> Ссылка [1/111] (страниц: 29)
    страница 1 из 29
    страница 2 из 29
    страница 3 из 29
    страница 4 из 29
    страница 5 из 29
   -> Сохранено 140 новых записей в cian_final_data.csv
    страница 6 из 29
    страница 7 из 29
    страница 8 из 29
    страница 9 из 29
    страница 10 из 29
   -> Сохранено 140 новых записей в cian_final_data.csv
    страница 11 из 29
    страница 12 из 29
    страница 13 из 29
    страница 14 из 29
    страница 15 из 29
   -> Сохранено 140 новых записей в cian_final_data.csv
    страница 16 из 29
    страница 17 из 29
    страница 18 из 29
    страница 19 из 29
    страница 20 из 29
   -> Сохранено 140 новых записей в cian_final_data.csv
    страница 21 из 29
    страница 22 из 29
    страница 23 из 29
    страница 24 из 29
    страница 25 из 29
   -> Сохранено 137 новых записей в cian_final_data.csv
    страница 26 из 29
    страница 27 из 29
    страница 28 из 29
    страница 29 из 29

>>> Ссылка [2/111] (страниц: 10)
    страница 1 из 10
  

In [112]:
df_urls = pd.read_csv("cian_final_data.csv")
print(f'старый файл: {df_urls.shape}')
df_urls = df_urls.drop_duplicates()
print(f'новый файл: {df_urls.shape}')
print(f'уникальных строк - {df_urls.url.nunique()}')

print('\n', 'дубликаты по url:', df_urls[df_urls.duplicated(subset=['url'], keep=False)])

FILENAME = "cian_urls_wo_duplicates.csv" 
df_urls.to_csv(FILENAME, mode='w', index=False, encoding='utf-8-sig')
print(f'Дубликаты удалены. Файл сохранен в {FILENAME}. Кол-во ссылок: {len(df_urls.url)}')


старый файл: (41762, 4)
новый файл: (38700, 4)
уникальных строк - 38700

 дубликаты по url: Empty DataFrame
Columns: [price, address, url, auction]
Index: []
Дубликаты удалены. Файл сохранен в cian_urls_wo_duplicates.csv. Кол-во ссылок: 38700


### Разведка данных - проверяем, какие атрибуты вообще бывают указаны на странице, чтобы хранить максимальное количество характеристик

In [72]:
import time
import pandas as pd
import os
import winsound
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

INPUT_FILE = "cian_urls_wo_duplicates.csv" #смотрим файл со ссылками, который создали ранее
LINKS_TO_CHECK = 20                #сколько проверяем ссылок

#собираем ссылки
try:
    df = pd.read_csv(INPUT_FILE)
    links = df['url'].dropna().tolist()
    print(f"Всего ссылок в файле: {len(links)}. Проверим первые {LINKS_TO_CHECK}.")
except FileNotFoundError:
    print(f"Файл {INPUT_FILE} не найден.")
    exit()

#запускаем браузер
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

#словарь для доступных полей
unique_fields = {}
#print(links)
#print(len(links))
#print('---------')
#print(links[20:LINKS_TO_CHECK])

try:
    
    for i, url in enumerate(links[:LINKS_TO_CHECK]):
        if "cian.ru" not in str(url):
            continue

        print(f"Заходим [{i+1}/{LINKS_TO_CHECK}]: {url}")
        
        try:
            driver.get(url)
            soup = BeautifulSoup(driver.page_source, 'html.parser')

            #все стандартные блоки атрибутов лежат в блоке OfferSummaryInfoItem
            items = soup.find_all('div', {'data-name': 'OfferSummaryInfoItem'})
            
            
            for item in items:
                texts = item.find_all('p')
                if len(texts) >= 2:
                    field_name = texts[0].get_text(strip=True)
                    example_value = texts[1].get_text(strip=True)
                    
                    #если поля нет в словаре пока, то добавляем его туда
                    if field_name not in unique_fields:
                        unique_fields[field_name] = example_value
                        print(f"    Новое поле: '{field_name}' (Пример: {example_value})")

        except Exception as e:
            print(f"Ошибка: {e}")
            continue

finally:
    driver.quit()

    print("\n" + "="*40)
    print("ИТОГОВЫЙ СПИСОК ДОСТУПНЫХ ПОЛЕЙ:")
    for name, example in unique_fields.items():
        print(f"- {name}: {example}")

Всего ссылок в файле: 39561. Проверим первые 20.
Заходим [1/20]: https://www.cian.ru/sale/flat/324994191/
    Новое поле: 'Тип жилья' (Пример: Вторичка / Апартаменты)
    Новое поле: 'Общая площадь' (Пример: 16 м²)
    Новое поле: 'Жилая площадь' (Пример: 10 м²)
    Новое поле: 'Площадь кухни' (Пример: 5 м²)
    Новое поле: 'Высота потолков' (Пример: 2,7 м)
    Новое поле: 'Санузел' (Пример: 1 раздельный)
    Новое поле: 'Вид из окон' (Пример: Во двор)
    Новое поле: 'Ремонт' (Пример: Евроремонт)
    Новое поле: 'Продаётся с мебелью' (Пример: Да)
    Новое поле: 'Тип дома' (Пример: Кирпичный)
    Новое поле: 'Несовершеннолетние собственники' (Пример: Нет)
    Новое поле: 'Материнский капитал при покупке' (Пример: Не использовался)
Заходим [2/20]: https://www.cian.ru/sale/flat/320317503/
    Новое поле: 'Парковка' (Пример: Наземная)
Заходим [3/20]: https://www.cian.ru/sale/flat/322151830/
Заходим [4/20]: https://www.cian.ru/sale/flat/324172390/
Заходим [5/20]: https://www.cian.ru/sale/

In [73]:
#этот кусок работает только с впн, иначе гугл блочит запрос
from googletrans import Translator

#создаем здесь себе список доступных атрибутов, заодно перееводим все на англ для названия полей
translator = Translator()

keys_list = list(unique_fields.keys())
values_list = list(unique_fields.values())

attributes = pd.DataFrame(zip(keys_list, 
                              [translator.translate(name, dest='en').text.lower().replace(' ', '_').replace('/','_') for name in keys_list],
                              values_list),
                          columns = ['field_name_ru', 'field_name_en', 'field_example'])
attributes

,field_name_ru,field_name_en,field_example
0,Тип жилья,housing_type,Вторичка / Апартаменты
1,Общая площадь,total_area,16 м²
2,Жилая площадь,living_area,10 м²
3,Площадь кухни,kitchen_area,5 м²
4,Высота потолков,ceiling_height,"2,7 м"
5,Санузел,bathroom,1 раздельный
6,Вид из окон,view_from_the_windows,Во двор
7,Ремонт,repair,Евроремонт
8,Продаётся с мебелью,sold_with_furniture,Да
9,Тип дома,home_type,Кирпичный


### Проходимся по каждому объявлению и вытаскиваем характеристики

In [144]:
import time
import pandas as pd
import re
import random
import json
import dateparser
import requests

from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

#-------------БЛОК ФУНКЦИЙ------------------------
#ищем координаты объекта
def get_coords_from_script(soup):
    scripts = soup.find_all('script')
    for script in scripts:
        if script.string:
            #ищем подстроку "coordinates":{"lat":55.123,"lng":37.123} - ищем все такие куски
            matches = re.findall(r'"coordinates":\{"lat":([\d\.]+),"lng":([\d\.]+)\}', script.string)
            for lat, lng in matches:
                #если нули, то ничего не возвращаем
                if float(lat) != 0 and float(lng) != 0:
                    return float(lat), float(lng)
    return None, None

#ищем ссылки и сохраняем их в нормальном формте
def get_photos_from_script(soup):
    scripts = soup.find_all('script')
    found_urls = []
    for script in scripts:
        if script.string and '"photos":[' in script.string:
            urls = re.findall(r'"fullUrl":"([^"]+)"', script.string)
            clean_urls = [u.replace("\\u002F", "/") for u in urls]
            found_urls.extend(clean_urls)
    
    return list(set(found_urls))

#строка с циана'вчера, 18:55' или '25 окт, 14:00' превращается в нормальный datetime
def parse_date_string(date_str):

    clean_str = str(date_str).replace("Обновлено", "").replace(":", "", 1).strip()
    
    try:
        dt = dateparser.parse(clean_str, languages=['ru'])
        
        if dt:
            return dt.strftime("%Y-%m-%d %H:%M:%S")
        else:
            return date_str
            
    except Exception:
        return date_str


def send_telegram_alert(message):
    #телега
    try:
        url = f"https://api.telegram.org/bot{TG_TOKEN}/sendMessage"
        data = {"chat_id": TG_CHAT_ID, "text": message}
        requests.post(url, data=data)
    except Exception as e:
        print(f"Не удалось отправить уведомление: {e}")


#-----Работа в файлах-----------
INPUT_FILE = "cian_urls_wo_duplicates.csv" #файл со ссылками
OUTPUT_FILE = "cian_extended_data.csv" #файл с расширенными данными

TG_TOKEN = "8323989130:AAG3ezUom-z5heFqH3Fms1wwA9K_HguLaEI"  
TG_CHAT_ID = 389535583

#если файла с данными нет, то создаем его на основе файла со ссылками
#если расширенный файл уже создан, то дополняем только то, что не заполнилось в предыдущие запуски
if os.path.exists(OUTPUT_FILE):
    print(f"Найден существующий файл {OUTPUT_FILE}.")
    df = pd.read_csv(OUTPUT_FILE, low_memory=False)
else:
    print(f"Файл {OUTPUT_FILE} не найден. Создаем новый на основе {INPUT_FILE}.")
    try:
        df = pd.read_csv(INPUT_FILE, low_memory=False)
    except FileNotFoundError:
        print(f"ОШИБКА: Исходный файл {INPUT_FILE} не найден!")
        exit()



#настройки браузера
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--log-level=3")
options.page_load_strategy = 'eager'

#пытаемся обойти блокировку vpn (это не сработало)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

prefs = {"profile.managed_default_content_settings.images": 2}
options.add_experimental_option("prefs", prefs)


driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
print("Браузер запущен")

#df = df.loc[[1677]]
#функция для поиска атрибутов в блоке OfferSummaryInfoItem
def get_cian_attribute(soup, label_text):
    all_items = soup.find_all('div', {'data-name': 'OfferSummaryInfoItem'})
    for item in all_items:
        texts = item.find_all('p')
        if len(texts) >= 2:
            #ищем названия полей + чистим от всяких нечитаемых символов
            name_raw = texts[0].get_text(strip=True)
            name_clean = " ".join(name_raw.split()) #"О\xa0подъезде"
            
            value_raw = texts[1].get_text(strip=True)
            value_clean = " ".join(value_raw.split())

        
            if label_text.lower() in name_clean.lower():
                return value_clean
    return None


#итоговый список колонок, полученный с помощью скриптов выше
new_cols = [
    'title', 'rooms_count',
    'housing_type', 'total_area', 'living_area', 'kitchen_area', 'floor_info', 
    'ceiling_height', 'bathroom', 'balcony_loggia', 'view_from_the_windows', 
    'repair', 'year_of_construction', 'construction_series', 'number_of_elevators', 
    'floor_type', 'entrances', 'about_the_entrance', 'parking', 'heating', 
    'accident_rate', 'gas_supply', 'sold_with_furniture', 'home_type', 
    'metro_info', 'highway_info', 'description', 'lat', 'lng', 
    'minor_owners', 'maternity_capital', 
    'publication_date_str', 'publication_date', 'photos_url', 'request_time'
]


for col in new_cols:
    if col not in df.columns:
        df[col] = None

#--------ОСНОВА-----
try:
    #сколько осталось обработать
    total_rows = len(df)
    #индексы строк, которые надо обработать впервые или заново, если возникла ошибка с ВПН
    rows_to_process = df[(df['request_time'].isna()) | (df['title']=='Кажется, у вас включён VPN') | (df['title'].isna())].index.tolist()
    
    print(f"\nВсего строк: {total_rows}, обработать осталось  {len(rows_to_process)}")

    for i, index in enumerate(rows_to_process):
        start_time = time.time()
        
        row = df.loc[index]
        url = row['url']
        address = row['address']
        
        if pd.isna(url) or "cian.ru" not in str(url):
            continue
            
        print(f"[Обработка {i+1}/{len(rows_to_process)}] (номер строки в файле – {index+1})")
        
        try:
            driver.get(url)
            #проставляем дату
            current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            '''
            #на случай капчи
            while True:
                page_title = driver.title.lower()
                #смотрим на заголовок страницы
                if "captcha" in page_title or "security" in page_title or "доступ ограничен" in driver.page_source:
                    print(f"!!! КАПЧА ОБНАРУЖЕНА !!! Скрипт на паузе.")
                    #вызывааем звук, если капча
                    try:
                        import winsound
                        winsound.Beep(1000, 500) 
                    except:
                        pass
                    time.sleep(10) #ждет 10 сек и вызывает проверку снова
                else:
                    #выходим, если все ок и капчи нет
                    break
            '''


            #задаем рандомное время прогрузки страницы, чтобы больше быть похожим на человека
            #sleep_time = random.uniform(1.5, 2.5)
            #time.sleep(sleep_time) 
    
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            
            #записываем название объявления и определяем комнатность
            h1_tag = soup.find('h1')
            title_text = h1_tag.get_text(strip=True) if h1_tag else "Не найден"

            sent_alert = False # Флаг, чтобы не спамить сообщениями каждую секунду
            
            while True:
                
                # Если сработала защита
                if "VPN" in title_text.upper() or "captcha" in title_text.lower():                    
                    # === ОТПРАВЛЯЕМ УВЕДОМЛЕНИЕ (Один раз) ===
                    if not sent_alert:
                        send_telegram_alert("🚨 Циан выдал КАПЧУ или ошибку VPN. Скрипт на паузе!")
                        sent_alert = True
                        
                    time.sleep(10)
                else:
                    if sent_alert:
                        send_telegram_alert("✅ Капча пройдена, работаю дальше.")
                    break
            
            rooms_count = None
            if h1_tag:
                #ищем цифру перед "-комн"
                rooms_match = re.search(r'(\d+)-комн', title_text, re.IGNORECASE)
                if rooms_match:
                    rooms_count = rooms_match.group(1)
                #студия
                elif "студия" in title_text.lower():
                    rooms_count = "Студия"
                #свободная планировка
                elif "свобод" in title_text.lower() and "планиров" in title_text.lower():
                    rooms_count = "Своб. планировка"
                    
            #собираем все атрибуты
            housing_type = get_cian_attribute(soup, "Тип жилья")
            total_area = get_cian_attribute(soup, "Общая площадь") or get_cian_attribute(soup, "Общая")
            living_area = get_cian_attribute(soup, "Жилая площадь")
            kitchen_area = get_cian_attribute(soup, "Площадь кухни")
            ceiling_height = get_cian_attribute(soup, "Высота потолков")
            bathroom = get_cian_attribute(soup, "Санузел")
            balcony_loggia = get_cian_attribute(soup, "Балкон/лоджия")
            view_from_the_windows = get_cian_attribute(soup, "Вид из окон")
            repair = get_cian_attribute(soup, "Ремонт")
            sold_with_furniture = get_cian_attribute(soup, "Продаётся с мебелью")
            year_of_construction = get_cian_attribute(soup, "Год постройки") or get_cian_attribute(soup, "Срок сдачи")
            construction_series = get_cian_attribute(soup, "Строительная серия")
            number_of_elevators = get_cian_attribute(soup, "Количество лифтов")
            floor_type = get_cian_attribute(soup, "Тип перекрытий")
            entrances = get_cian_attribute(soup, "Подъезды")
            about_the_entrance = get_cian_attribute(soup, "О подъезде")
            parking = get_cian_attribute(soup, "Парковка")
            heating = get_cian_attribute(soup, "Отопление")
            accident_rate = get_cian_attribute(soup, "Аварийность")
            gas_supply = get_cian_attribute(soup, "Газоснабжение")
            home_type = get_cian_attribute(soup, "Тип дома")
            minor_owners = get_cian_attribute(soup, "Несовершеннолетние собственники")
            maternity_capital = get_cian_attribute(soup, "Материнский капитал при покупке")

            #определям этаж
            floor_info = get_cian_attribute(soup, "Этаж") 
            if not floor_info:
                try:
                    floor_labels = soup.find_all(string=re.compile(r"^Этаж$"))
                    for label in floor_labels:
                        parent = label.parent
                        next_elem = parent.find_next_sibling()
                        if next_elem:
                            text_val = next_elem.get_text(strip=True)
                            if re.search(r'\d', text_val):
                                floor_info = text_val
                                break
                except:
                    pass
            if not floor_info:
                h1_tag = soup.find('h1')
                if h1_tag:
                    floor_regex = re.search(r'(\d+)\s*(?:/|из)\s*(\d+)', h1_tag.get_text())
                    if floor_regex:
                        floor_info = f"{floor_regex.group(1)} из {floor_regex.group(2)}"
            if not floor_info: floor_info = "Не найден"

            #метро и описание
            metro_block = soup.find('ul', {'data-name': 'UndergroundList'})
            metro_info = None
            
            if metro_block:
                metro_items = []
                #перебираем каждый пункт метро отдельно (теги <li>)
                for item in metro_block.find_all('li'):
                    #чистый текст: "Автозаводская 5 мин."
                    text = item.get_text(strip=True, separator=" ")
                    
                    #превращаем весь HTML пункта в строку и ищем ключевые слова в названиях классов/иконок
                    item_html = str(item).lower()
                    
                    #ищем иконку машины
                    if "m14 7-.84" in item_html:
                        transport_type = "на машине"
                    #по умолчанию пешком
                    else:
                        transport_type = "пешком"
                        
                    #сбираем обратно в строку
                    if "Посмотреть ещё" in text:
                        continue
                    metro_items.append(f"{text} ({transport_type})")
                
                #все объединяем
                metro_info = ", ".join(metro_items)

            #highway
            highway_block = soup.find('ul', {'data-name': 'HighwayList'})
            highway_info = highway_block.get_text(separator=", ", strip=True) if highway_block else None

            
            desc_block = soup.find('div', {'data-name': 'Description'})
            description = desc_block.get_text(separator=" ", strip=True) if desc_block else None

            #дата публикации объявления по фразе "Обновлено"
            pub_date_str = "Не найдена"
            date_node = soup.find(string=re.compile(r"Обновлено"))
            if date_node:
                pub_date_str = date_node.strip()

            pub_date = parse_date_string(pub_date_str)
            
            #вытаскиваем первые 15 ссылок на фотки
            raw_photos = get_photos_from_script(soup)
            photos_str = ", ".join(raw_photos[:15])

            #координаты
            lat, lng = get_coords_from_script(soup)

            #записываем все полчившеееся
            df.at[index, 'title'] = title_text
            df.at[index, 'rooms_count'] = rooms_count
            
            df.at[index, 'housing_type'] = housing_type
            df.at[index, 'total_area'] = total_area
            df.at[index, 'living_area'] = living_area
            df.at[index, 'kitchen_area'] = kitchen_area
            df.at[index, 'floor_info'] = floor_info 
            df.at[index, 'ceiling_height'] = ceiling_height
            df.at[index, 'bathroom'] = bathroom
            df.at[index, 'balcony_loggia'] = balcony_loggia
            df.at[index, 'view_from_the_windows'] = view_from_the_windows
            df.at[index, 'repair'] = repair
            df.at[index, 'year_of_construction'] = year_of_construction
            df.at[index, 'construction_series'] = construction_series
            df.at[index, 'number_of_elevators'] = number_of_elevators
            df.at[index, 'floor_type'] = floor_type
            df.at[index, 'entrances'] = entrances
            df.at[index, 'about_the_entrance'] = about_the_entrance
            df.at[index, 'parking'] = parking
            df.at[index, 'heating'] = heating
            df.at[index, 'accident_rate'] = accident_rate
            df.at[index, 'gas_supply'] = gas_supply
            df.at[index, 'sold_with_furniture'] = sold_with_furniture
            df.at[index, 'home_type'] = home_type
            df.at[index, 'minor_owners'] = minor_owners
            df.at[index, 'maternity_capital'] = maternity_capital

            
            df.at[index, 'metro_info'] = metro_info
            df.at[index, 'highway_info'] = highway_info
            
            df.at[index, 'description'] = description
            df.at[index, 'lat'] = lat
            df.at[index, 'lng'] = lng
            df.at[index, 'publication_date_str'] = pub_date_str
            df.at[index, 'publication_date'] = pub_date
            df.at[index, 'photos_url'] = photos_str
            df.at[index, 'request_time'] = current_time

            
            end_time = time.time()
            execution_time = end_time - start_time
            print(f"--> Время обработки объявления: {execution_time:.2f} сек.")
            
            df.at[index, 'process_time'] = round(execution_time, 2)

            #сохраняем инфу каждые 5 объявлений, чтобы иметь возможность перезапустить скрипт при ошибке
            if (i + 1) % 5 == 0:
                df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
                print(f"   [Сохранено {i+1} строк]")


        except Exception as e:
            print(f"Ошибка: {e}")
            continue

finally:
    driver.quit()
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"\nГОТОВО! Сохранено в {OUTPUT_FILE}")

Найден существующий файл cian_extended_data.csv.
Браузер запущен

Всего строк: 38700, обработать осталось  418
[Обработка 1/418] (номер строки в файле – 35516)
--> Время обработки объявления: 2.45 сек.
[Обработка 2/418] (номер строки в файле – 35531)


C:\Users\munir\AppData\Local\Temp\ipykernel_18720\195610640.py:343: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'year_of_construction'] = year_of_construction
C:\Users\munir\AppData\Local\Temp\ipykernel_18720\195610640.py:347: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'entrances'] = entrances


--> Время обработки объявления: 1.21 сек.
[Обработка 3/418] (номер строки в файле – 35601)
--> Время обработки объявления: 0.80 сек.
[Обработка 4/418] (номер строки в файле – 35602)
--> Время обработки объявления: 0.71 сек.
[Обработка 5/418] (номер строки в файле – 35603)
--> Время обработки объявления: 0.82 сек.
   [Сохранено 5 строк]
[Обработка 6/418] (номер строки в файле – 35604)
--> Время обработки объявления: 1.08 сек.
[Обработка 7/418] (номер строки в файле – 35621)
--> Время обработки объявления: 0.85 сек.
[Обработка 8/418] (номер строки в файле – 35641)
--> Время обработки объявления: 0.71 сек.
[Обработка 9/418] (номер строки в файле – 35642)
--> Время обработки объявления: 0.76 сек.
[Обработка 10/418] (номер строки в файле – 35643)
--> Время обработки объявления: 0.85 сек.
   [Сохранено 10 строк]
[Обработка 11/418] (номер строки в файле – 35644)
--> Время обработки объявления: 0.96 сек.
[Обработка 12/418] (номер строки в файле – 35645)
--> Время обработки объявления: 0.93 сек

In [17]:
for index1, row in attributes.iterrows():
    print(f'df.at[index, "{row["field_name_en"]}"] = {row["field_name_en"]}')

df.at[index, "housing_type"] = housing_type
df.at[index, "total_area"] = total_area
df.at[index, "living_area"] = living_area
df.at[index, "kitchen_area"] = kitchen_area
df.at[index, "ceiling_height"] = ceiling_height
df.at[index, "repair"] = repair
df.at[index, "sold_with_furniture"] = sold_with_furniture
df.at[index, "home_type"] = home_type
df.at[index, "parking"] = parking
df.at[index, "bathroom"] = bathroom
df.at[index, "year_of_construction"] = year_of_construction
df.at[index, "number_of_elevators"] = number_of_elevators
df.at[index, "floor_type"] = floor_type
df.at[index, "entrances"] = entrances
df.at[index, "heating"] = heating
df.at[index, "accident_rate"] = accident_rate
df.at[index, "balcony_loggia"] = balcony_loggia
df.at[index, "view_from_the_windows"] = view_from_the_windows
df.at[index, "gas_supply"] = gas_supply
df.at[index, "construction_series"] = construction_series
df.at[index, "about_the_entrance"] = about_the_entrance


In [117]:
import pandas as pd
file_data = "cian_extended_data.csv"
df_cian = pd.read_csv(file_data, low_memory=False)
df_new = df_cian.drop_duplicates()
df_new

,price,address,url,auction,title,rooms_count,housing_type,total_area,living_area,kitchen_area,...,highway_info,description,lat,lng,minor_owners,maternity_capital,publication_date_str,publication_date,photos_url,request_time
0,5350000,"Москва, ЮАО, р-н Даниловский, м. Автозаводская...",https://www.cian.ru/sale/flat/324994191/,False,"Продается апартаменты-студия, 16 м²",Студия,Вторичка / Апартаменты,16 м²,10 м²,5 м²,...,NaN,АПАРТАМЕНТЫ:\nПродаётся светлая и аккуратная с...,55.703885,37.652768,Нет,Не использовался,"Обновлено: вчера, 12:29",2025-12-19 12:29:00,https://images.cdn-cian.ru/images/kvartira-mos...,2025-12-20 03:40:29
1,5990000,"Москва, САО, р-н Головинский, м. Моссельмаш, С...",https://www.cian.ru/sale/flat/320317503/,False,"Продается апартаменты-студия, 16 м²",Студия,Вторичка / Апартаменты,16 м²,"13,5 м²",NaN,...,NaN,Уютная студия с новым ремонтом и мебелью в ЖК ...,55.859322,37.515874,NaN,NaN,"Обновлено: вчера, 16:25",2025-12-19 16:25:00,https://images.cdn-cian.ru/images/kvartira-mos...,2025-12-20 03:40:32
2,5380000,"Москва, ВАО, р-н Вешняки, м. Новогиреево, Вешн...",https://www.cian.ru/sale/flat/324490616/,True,"Продается 2-комн. квартира, 39 м²",2,Вторичка,39 м²,NaN,NaN,...,NaN,"Жилое помещение (квартира), кад. 77:03:000700...",55.740066,37.822882,NaN,NaN,"Обновлено: вчера, 10:40",2025-12-19 10:40:00,https://images.cdn-cian.ru/images/2718616692-1...,2025-12-20 03:40:35
3,5990000,"Москва, САО, р-н Головинский, м. Моссельмаш, С...",https://www.cian.ru/sale/flat/322151830/,False,"Продается апартаменты-студия, 15,2 м²",Студия,Вторичка / Апартаменты,"15,2 м²","12,7 м²",NaN,...,NaN,Уютная студия с новым ремонтом и мебелью в ЖК ...,55.859322,37.515874,NaN,NaN,"Обновлено: вчера, 19:55",2025-12-19 19:55:00,https://images.cdn-cian.ru/images/kvartira-mos...,2025-12-20 03:40:38
4,5500000,"Москва, ТАО (Троицкий), пос. Рогово, улица Шко...",https://www.cian.ru/sale/flat/323600775/,False,"Продается 1-комн. квартира, 32,6 м²",1,Вторичка,"32,6 м²",NaN,"7,5 м²",...,"Калужское шоссе, 59 км от МКАД, Симферопольско...",Номер в базе: 10289395.\n\n Московская прописк...,55.212647,37.072573,NaN,NaN,"Обновлено: вчера, 18:22",2025-12-19 18:22:00,https://images.cdn-cian.ru/images/2690280423-1...,2025-12-20 03:40:41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38695,2813324500,"Москва, ЦАО, р-н Пресненский, м. Баррикадная, ...",https://www.cian.ru/sale/flat/284989203/,False,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38696,1328522000,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",https://www.cian.ru/sale/flat/322204297/,False,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38697,1328522000,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",https://www.cian.ru/sale/flat/321207818/,False,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38698,1732170835,"Москва, ЦАО, р-н Пресненский, м. Охотный ряд, ...",https://www.cian.ru/sale/flat/300661842/,False,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
import pandas as pd
file_data = "cian_extended_data.csv"
df_cian = pd.read_csv(file_data, low_memory=False)
df_cian.url.nunique()

#анализируем длины полей в датасете
lengths = []
for col in df_cian.columns:
    max_len = df_cian[col].astype(str).str.len().max()
    lengths.append({'column': col,
                    'max_length': max_len,
                    'col_type': df_cian[col].dtype})
lengths_df = pd.DataFrame(lengths)
lengths_df

,column,max_length,col_type
0,price,10,int64
1,address,123,object
2,url,40,object
3,auction,5,bool
4,title,53,object
5,rooms_count,16,object
6,housing_type,25,object
7,total_area,9,object
8,living_area,8,object
9,kitchen_area,8,object


In [39]:
import pandas as pd
import re
import numpy as np

# ВТОРИЧКИ
def parse_data_with_patterns(text,pattern1,pattern2, numbers : bool):

    if pd.isna(text):
        return pd.Series([np.nan, np.nan])
    
    text = str(text).strip().lower()

    sovm = np.nan
    razdel = np.nan

    sovm_match = re.search(pattern1, text)
    razdel_match = re.search(pattern2, text)

    if numbers:
      if sovm_match:
          sovm = int(sovm_match.group(1))
    
      if razdel_match:
          razdel = int(razdel_match.group(1))
    
      return pd.Series([sovm, razdel])

    else:
      if sovm_match:
        sovm = 'Yes'
      if razdel_match:
        razdel = 'Yes'
      return pd.Series([sovm, razdel])

def clean_rooms_count(row):
    
    title = str(row['title']).lower()
    old_rooms = str(row['rooms_count'])
    
    #ищем долю
    ratio_pattern = r'доля|%|\d+\s*/\s*\d+'
    if re.search(ratio_pattern, title):
        return 'Доля'
        
    #многокомнатную
    if 'многоком' in title:
        return 'Многокомнатная'
        
    #перезапись свободной планировки
    if 'своб' in old_rooms.lower():
        return 'Свободная'

    return old_rooms


file_data = "cian_extended_data.csv"
df_cian = pd.read_csv(file_data, low_memory=False)


#удаляем те объявления, которые были сняты с публикации в процессе парсинга
df_cian_clean = df_cian[(df_cian['title'] != 'Страница не найдена') & (df_cian['title'] != 'Не найден')]
df_cian_clean = df_cian_clean.dropna(subset=['price'])
print(f'Удалено строк: {df_cian.url.nunique() - df_cian_clean.url.nunique()}')

#price в инт
df_cian_clean['price'] = df_cian_clean['price'].astype('Int64')



#дозаполняю доли и многокомнатные, Своб.планировка привожу к виду "Свободная"
df_cian_clean['rooms_count'] = df_cian_clean.apply(clean_rooms_count, axis=1)


#вторичка и апартаменты с пентхаусами
df_cian_clean['new_building'] = df_cian_clean['housing_type'].apply(lambda x: 'Новостройка' if 'новостр' in str(x).lower() else
                                                                              'Вторичка' if 'вторич' in str(x).lower() else np.nan)
df_cian_clean['apartment'] = df_cian_clean['housing_type'].apply(lambda x: 'Апартаменты' if 'апартам' in str(x).lower() else
                                                                            'Пентхаус' if 'пентх' in str(x).lower() else np.nan)
#del df_cian_clean['housing_type']


#всякие квадратные метры
df_cian_clean['total_area'] = df_cian_clean['total_area'].str.replace('м²','') \
                             .str.replace(',','.') \
                             .astype(float)
df_cian_clean['living_area'] = df_cian_clean['living_area'].str.replace('м²','') \
                             .str.replace(',','.') \
                             .astype(float)
df_cian_clean['kitchen_area'] = df_cian_clean['kitchen_area'].str.replace('м²','') \
                             .str.replace(',','.') \
                             .astype(float)
df_cian_clean['ceiling_height'] = df_cian_clean['ceiling_height'].str.replace('м','') \
                             .str.replace(',','.') \
                             .astype(float)


#этажность
df_cian_clean[['floor', 'total_floors']] = df_cian_clean['floor_info'].str.split(' из ', expand=True)
df_cian_clean['floor'] = df_cian_clean['floor'].astype(int)
df_cian_clean['total_floors'] = df_cian_clean['total_floors'].astype(int)
#del df_cian_clean['floor_info']

#санузлы
comb_bathroom_pattern = r'(\d+)\s*[-–]?\s*совмещен'
razdel_pattern = r'(\d+)\s*[-–]?\s*раздел'
df_cian_clean[['combined_bathroom', 'separated_bathroom']] = df_cian_clean['bathroom'].apply(lambda x: parse_data_with_patterns(x,comb_bathroom_pattern,razdel_pattern,True))
#del cian_cleaned['bathroom']

#балкон лоджия
balcony_pattern = r'(\d+)\s*[-–]?\s*балко'
loggia_pattern = r'(\d+)\s*[-–]?\s*лодж'
df_cian_clean[['balcony', 'loggia']] = df_cian_clean['balcony_loggia'].apply(lambda x: parse_data_with_patterns(x,balcony_pattern,loggia_pattern,True))
#del cian_cleaned['balcony_loggia']

#лифты
passanger_pattern = r'(\d+)\s*[-–]?\s*пассажир'
cargo_pattern = r'(\d+)\s*[-–]?\s*грузов'
df_cian_clean[['passenger_elevator', 'cargo_elevator']] = df_cian_clean['number_of_elevators'].apply(lambda x: parse_data_with_patterns(x,passanger_pattern,cargo_pattern,True))
#del df_cian_clean['number_of_elevators']

#тип перекрытий "нет информации" заменяем на Nan
df_cian_clean['floor_type'] = df_cian_clean['floor_type'].apply(lambda x: np.nan if 'нет информации' in str(x).lower() else x)

#всякое к int
df_cian_clean['entrances'] = df_cian_clean['entrances'].astype('Int64')
df_cian_clean['year_of_construction'] = df_cian_clean['year_of_construction'].astype('Int64')


#мусоропровод и консьерж
garbage_pattern = 'мусор'
concierge_pattern = 'консь'
df_cian_clean[['garbage_chute', 'concierge']] = df_cian_clean['about_the_entrance'].apply(lambda x: parse_data_with_patterns(x,garbage_pattern,concierge_pattern,False))
#del cian_cleaned['about_the_entrance']

#всякое к Д/Н
df_cian_clean['auction'] = df_cian_clean['auction'].apply(lambda x: 'Д' if x == True else 'Н')
df_cian_clean['garbage_chute'] = df_cian_clean['garbage_chute'].apply(lambda x: 'Д' if str(x).lower() == 'yes' else 
                                                                                'Н' if str(x).lower() == 'no' else np.nan)
df_cian_clean['concierge'] = df_cian_clean['concierge'].apply(lambda x: 'Д' if str(x).lower() == 'yes' else 
                                                                        'Н' if str(x).lower() == 'no' else np.nan)
df_cian_clean['accident_rate'] = df_cian_clean['accident_rate'].apply(lambda x: 'Д' if str(x).lower() == 'да' else
                                                                                'Н' if str(x).lower() == 'нет' else np.nan)
df_cian_clean['sold_with_furniture'] = df_cian_clean['sold_with_furniture'].apply(lambda x: 'Д' if str(x).lower() == 'да' else 
                                                                                            'Н' if str(x).lower() == 'нет' else np.nan)
df_cian_clean['minor_owners'] = df_cian_clean['minor_owners'].apply(lambda x: 'Д' if str(x).lower() == 'есть' else
                                                                              'Н' if str(x).lower() == 'нет' else np.nan)
df_cian_clean['maternity_capital'] = df_cian_clean['maternity_capital'].apply(lambda x: 'Д' if str(x).lower() == 'использовался' else
                                                                                        'Н' if str(x).lower() == 'не использовался' else np.nan)

#datetime
df_cian_clean['publication_date'] = pd.to_datetime(df_cian_clean['publication_date'])
df_cian_clean['request_time'] = pd.to_datetime(df_cian_clean['request_time'])


#переименую столбец
df_cian_clean = df_cian_clean.rename(columns={
    'home_type': 'house_type'
})



df_cian_clean.head()
'''df_cian_clean[(df_cian_clean['title'] == 'Продается 5/16 квартиры, 45/27,1/6 м²') |
              (df_cian_clean['title'] == 'Продается 10% квартиры, 33/27 м²') |
              (df_cian_clean['title'] == 'Продается доля, 58 м²')]'''
df_cian_clean.head(20)
df_cian_clean[df_cian_clean.address =='Москва, ЮВАО, р-н Некрасовка, м. Некрасовка, Рождественская улица, 21к5']    
df_cian_clean.info()

#порядок столбцов
df_cian_clean = df_cian_clean[['title',	'url',	'address',	'price', 'auction',	'rooms_count',	
                               #'housing_type', #check
                               'new_building',	
                               'apartment',	'total_area',	'living_area',	'kitchen_area',	'ceiling_height',
                               #'floor_info', #check
                               'floor',	'total_floors',
                               #'bathroom', #check
                               'combined_bathroom',	'separated_bathroom',
                               #'balcony_loggia', #check
                               'balcony', 'loggia',	'view_from_the_windows',	'repair',	'year_of_construction',	'construction_series',
                               #'number_of_elevators', #сheck
                               'passenger_elevator',	'cargo_elevator',	'floor_type',	'entrances',
                               #'about_the_entrance', #check
                               'garbage_chute',
                               'concierge',	'parking',	'heating',	'accident_rate', 'gas_supply', 'sold_with_furniture',
                               'house_type', 'metro_info', 'highway_info', 'description', 'lat',	'lng',
                               'minor_owners',	'maternity_capital', 'publication_date', 'request_time',	'photos_url'
]]
df_cian_clean.to_csv('cian_data_secondary_cleaned.csv',mode='w',index=False,header=True, encoding='utf-8-sig', sep=';')


Удалено строк: 24
<class 'pandas.core.frame.DataFrame'>
Index: 38676 entries, 0 to 38699
Data columns (total 52 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   price                  38676 non-null  Int64         
 1   address                38676 non-null  object        
 2   url                    38676 non-null  object        
 3   auction                38676 non-null  object        
 4   title                  38676 non-null  object        
 5   rooms_count            38676 non-null  object        
 6   housing_type           38676 non-null  object        
 7   total_area             38676 non-null  float64       
 8   living_area            28424 non-null  float64       
 9   kitchen_area           35043 non-null  float64       
 10  floor_info             38676 non-null  object        
 11  ceiling_height         26855 non-null  float64       
 12  bathroom               34472 non-null  object  

In [77]:
parse_date_string('Обновлено: сегодня, 12:29')

'2025-12-19 12:29:00'

In [21]:

stats = df_cian.groupby('home_type', as_index=False).agg({'url': 'count'})
stats
result = ", ".join([f"{row['home_type']} - {row['url']:,}".replace(',', ' ') for _, row in stats.iterrows()])
print(result)

Блочный - 1 709, Деревянный - 4, Кирпичный - 6 478, Монолитно-кирпичный - 1 431, Монолитный - 10 601, Панельный - 7 529, Сталинский - 53


In [35]:
df_cian_clean.head(20)


,title,url,address,price,auction,rooms_count,new_building,apartment,total_area,living_area,...,metro_info,highway_info,description,lat,lng,minor_owners,maternity_capital,publication_date,request_time,photos_url
0,"Продается апартаменты-студия, 16 м²",https://www.cian.ru/sale/flat/324994191/,"Москва, ЮАО, р-н Даниловский, м. Автозаводская...",5350000,Н,Студия,Вторичка,Апартаменты,16.0,10.0,...,"Автозаводская 5 мин. (пешком), ЗИЛ 9 мин. (пеш...",NaN,АПАРТАМЕНТЫ:\nПродаётся светлая и аккуратная с...,55.703885,37.652768,Н,Н,2025-12-19 12:29:00,2025-12-20 03:40:29,https://images.cdn-cian.ru/images/kvartira-mos...
1,"Продается апартаменты-студия, 16 м²",https://www.cian.ru/sale/flat/320317503/,"Москва, САО, р-н Головинский, м. Моссельмаш, С...",5990000,Н,Студия,Вторичка,Апартаменты,16.0,13.5,...,"Моссельмаш 10 мин. (пешком), Грачёвская 19 мин...",NaN,Уютная студия с новым ремонтом и мебелью в ЖК ...,55.859322,37.515874,Н,Н,2025-12-19 16:25:00,2025-12-20 03:40:32,https://images.cdn-cian.ru/images/kvartira-mos...
2,"Продается 2-комн. квартира, 39 м²",https://www.cian.ru/sale/flat/324490616/,"Москва, ВАО, р-н Вешняки, м. Новогиреево, Вешн...",5380000,Н,2,Вторичка,NaN,39.0,NaN,...,"Новогиреево 17 мин. (пешком), Новогиреево 5 ми...",NaN,"Жилое помещение (квартира), кад. 77:03:000700...",55.740066,37.822882,Н,Н,2025-12-19 10:40:00,2025-12-20 03:40:35,https://images.cdn-cian.ru/images/2718616692-1...
3,"Продается апартаменты-студия, 15,2 м²",https://www.cian.ru/sale/flat/322151830/,"Москва, САО, р-н Головинский, м. Моссельмаш, С...",5990000,Н,Студия,Вторичка,Апартаменты,15.2,12.7,...,"Моссельмаш 10 мин. (пешком), Грачёвская 19 мин...",NaN,Уютная студия с новым ремонтом и мебелью в ЖК ...,55.859322,37.515874,Н,Н,2025-12-19 19:55:00,2025-12-20 03:40:38,https://images.cdn-cian.ru/images/kvartira-mos...
4,"Продается 1-комн. квартира, 32,6 м²",https://www.cian.ru/sale/flat/323600775/,"Москва, ТАО (Троицкий), пос. Рогово, улица Шко...",5500000,Н,1,Вторичка,NaN,32.6,NaN,...,NaN,"Калужское шоссе, 59 км от МКАД, Симферопольско...",Номер в базе: 10289395.\n\n Московская прописк...,55.212647,37.072573,Н,Н,2025-12-19 18:22:00,2025-12-20 03:40:41,https://images.cdn-cian.ru/images/2690280423-1...
5,"Продается студия, 20 м²",https://www.cian.ru/sale/flat/324712040/,"Москва, ЮВАО, р-н Некрасовка, м. Некрасовка, П...",5600000,Н,Студия,Вторичка,NaN,20.0,12.0,...,"Некрасовка 3 мин. (пешком), Лухмановская 18 ми...","Новорязанское шоссе, 12 км от МКАД, Косинское ...",Код объекта: 1958895.\nУникальное предложение ...,55.704996,37.922335,Н,Н,2025-12-18 18:26:00,2025-12-20 03:40:44,https://images.cdn-cian.ru/images/2725644747-1...
6,"Продается 1-комн. квартира, 30,8 м²",https://www.cian.ru/sale/flat/323240964/,"Москва, ТАО (Троицкий), пос. ЛМС, Центральный ...",5700000,Н,1,Вторичка,NaN,30.8,NaN,...,NaN,"Калужское шоссе, 42 км от МКАД, Варшавское шос...",Номер в базе: 10099003.\n\n Продается светлая ...,55.314407,37.176553,Н,Н,2025-12-19 17:45:00,2025-12-20 03:40:47,https://images.cdn-cian.ru/images/2677754356-1...
7,"Продается апартаменты-студия, 31,9 м²",https://www.cian.ru/sale/flat/324212999/,"Москва, ВАО, р-н Измайлово, м. Измайловская, 4...",5490000,Н,Студия,Вторичка,Апартаменты,31.9,NaN,...,"Измайловская 12 мин. (пешком), Первомайская 12...",NaN,Номер в базе: 10409187.\n\n Продаю квартиру-ст...,55.794593,37.786240,Н,Н,2025-12-19 17:43:00,2025-12-20 03:40:49,https://images.cdn-cian.ru/images/2709752757-1...
8,"Продается студия, 35 м²",https://www.cian.ru/sale/flat/324559337/,"Москва, ТАО (Троицкий), м. Троицк, с. Былово, ...",5800000,Н,Студия,Вторичка,NaN,35.0,18.0,...,"Троицк откроется в 2026 15 мин. (на машине), В...","Калужское шоссе, 27 км от МКАД, Киевское шоссе...",На продажу выставлена современная квартира-сту...,55.448393,37.245634,Н,Н,2025-12-19 05:55:00,2025-12-20 03:40:52,https://images.cdn-cian.ru/images/2722324218-1...
9,"Продается апартаменты-студия, 12,7 м²",https://www.cian.ru/sale/flat/321742295/,"Москва, САО, р-н Ховрино, м. Грачёвская, Кли

In [ ]:

file_data_first = "cian_extended_data.csv"
file_data_second = ''
df_cian = pd.read_csv(file_data, low_memory=False)

In [ ]:
# merge новостроек и вторичек
import pandas as pd
import numpy as np

file_data_first = "cian_data_filtered.csv"
file_data_second = 'cian_data_secondary_cleaned.csv'
df_cian_first = pd.read_csv(file_data_first, low_memory=False, sep=';', encoding='cp1251')
df_cian_second = pd.read_csv(file_data_second, low_memory=False, sep=';')

#доп обработка для новостроек
#price в инт
df_cian_first['price'] = df_cian_first['price'].astype('Int64')
#в новостройках только не аукцион
df_cian_first['auction'] = 'Н'

df_cian_first['housing_type'] = df_cian_first['new_building'].fillna('').astype(str) + df_cian_first['apartment'].fillna('').astype(str)
df_cian_first['apartment'] = df_cian_first['housing_type'].apply(lambda x: 'Апартаменты' if 'апартам' in str(x).lower() else
                                                                            'Пентхаус' if 'пентх' in str(x).lower() else np.nan)
df_cian_first['new_building'] = df_cian_first['housing_type'].apply(lambda x: 'Новостройка' if 'новостр' in str(x).lower() else
                                                                              'Вторичка' if 'вторич' in str(x).lower() else np.nan)
del df_cian_first['housing_type']

#корректируем Д/Н
df_cian_first['garbage_chute'] = df_cian_first['garbage_chute'].apply(lambda x: 'Д' if str(x).lower() == 'yes' else 
                                                                                'Н' if str(x).lower() == 'no' else np.nan)
df_cian_first['concierge'] = df_cian_first['concierge'].apply(lambda x: 'Д' if str(x).lower() == 'yes' else 
                                                                        'Н' if str(x).lower() == 'no' else np.nan)
df_cian_first['minor_owners'] = df_cian_first['minor_owners'].apply(lambda x: 'Д' if str(x).lower() == 'есть' else
                                                                              'Н' if str(x).lower() == 'нет' else np.nan)
df_cian_first['maternity_capital'] = df_cian_first['maternity_capital'].apply(lambda x: 'Д' if str(x).lower() == 'использовался' else
                                                                                        'Н' if str(x).lower() == 'не использовался' else np.nan)
df_cian_first['ramp'] = df_cian_first['ramp'].apply(lambda x: 'Д' if str(x).lower() == 'есть' else
                                                              'Н' if str(x).lower() == 'нет' else np.nan)

#округляем дату до минут
#datetime
df_cian_first['publication_date'] = pd.to_datetime(df_cian_first['publication_date'], dayfirst=True)
df_cian_second['publication_date'] = pd.to_datetime(df_cian_second['publication_date'], dayfirst=False)
df_cian_first['request_time'] = pd.to_datetime(df_cian_first['request_time'], dayfirst=True)
df_cian_second['request_time'] = pd.to_datetime(df_cian_second['request_time'], dayfirst=False)

del df_cian_first['offer_date']

#оставляем только соотв тип строительства
df_cian_first = df_cian_first[df_cian_first['new_building'] == 'Новостройка']
df_cian_second = df_cian_second[df_cian_second['new_building'] == 'Вторичка']

#вручную исправляю строку, которая не распарсилась
df_cian_second.loc[df_cian_second['title'] == 'Продается 0,225 квартиры, 35 м²', 'rooms_count'] = 'Доля'
del df_cian_second['title']

df_merged = pd.concat([df_cian_second, df_cian_first], ignore_index=True)
#df_cian_second.info()

'''
for col in df_cian_first.columns:
    type_1 = df_cian_first[col].dtype
    
    if col in df_cian_second.columns:
        type_2 = df_cian_second[col].dtype
    else:
        type_2 = "НЕТ В SEC"
        
    print(f'{col:<30} | {str(type_1):<15} | {str(type_2):<15}')
'''


df_merged.info()

df_merged = df_merged[df_merged['auction'] == 'Н']
df_merged.to_csv('cian_full_data.csv',mode='w',index=False,header=True, encoding='utf-8-sig', sep=';')
df_merged.head()

df_cian_first.info()

for col in df_cian_first.columns:
    print(col)

print('----')
for col in df_cian_second.columns:
    print(col)
   

In [ ]:
df_merged.shape

In [ ]:
print(df_cian_first.shape, df_cian_first.url.nunique())
print(df_cian_second.shape, df_cian_second.url.nunique())
print(df_merged.shape, df_merged.url.nunique())


In [ ]:
df_cian_first

In [ ]:
duplicates = df_merged[df_merged.duplicated(subset=['url'], keep=False)]

#сортируем по URL, чтобы одинаковые квартиры стояли рядом
duplicates_sorted = duplicates.sort_values(by='url')

duplicates_sorted
duplicates_sorted.to_csv('duplicates_sorted.csv',mode='w',index=False,header=True, encoding='utf-8-sig', sep=';')
